# rl19_report

> Phase 19 (`specs/multiagent/phase19/`): the analysis under `RL19_PLAN`. `rl19Summaries` computes every run's measures (`rl19_measures`, `rl19_spec`), H19-1 and H19-2 (one-sided exact permutation tests, 8 runs per arm, Holm within the primary family), the H19-3 constraint (a hierarchical bootstrap interval, raw welfare beside it), H19-4 (the kill criterion's sign-flip over arm 4's runs, with the slope over team size, persistence and recovery), H19-5 and H19-6 with intervals, Gate A1, the kill criterion, the best arm and halts by cause; it refuses rows unless every run covers the grid. `writeRL19` (every plan item reported), `figRL19` (`rl19.png`) and `syntheticRows19` (known effects, for tests and the quick demo).

In [ ]:
#| default_exp rl19_report

In [ ]:
#| hide
from nbdev.showdoc import *

## Summaries and tests

In [ ]:
#| export
import json

import numpy as np

from explRL.bench_stats import holm
from explRL.rl16_report import hierBoot, permTest, signFlip
from explRL.rl18_report import _boot
from explRL.rl19_grid import ARMS19, RL19_PLAN, RUNS19
from explRL.rl19_measures import NAN, coverage, index, runMeasures
from explRL.rl19_spec import specMeasures

ARM_LABEL = dict(a19credit='1 credit', a19contest='2 contest, varied', a19fixed='3 contest, fixed 4', a19open='4 open')


def runNames19(arm, runs=RUNS19):
    return [f'{a}_s{s}' for a, s in runs if a == arm]


def _m(v):
    v = [x for x in v if np.isfinite(x)]
    return float(np.mean(v)) if v else NAN


def haltCauses(halts):
    "{cause: runs} from {run: halted.json reason or None}: invalid replies, KL, answers, jump, or none."
    out = {}
    for run, why in sorted((halts or {}).items()):
        c = 'none' if not why else next((k for k in ('answers', 'jump', 'KL', 'invalid') if k in why), 'other')
        out.setdefault(c, []).append(run)
    return out


def rl19Summaries(rows, plan=RL19_PLAN, runs=RUNS19, nboot=2000, warm='warm', seeds=10, halts=None, nNull=200):
    """RL19_PLAN's analysis over the evaluation rows (contestSuite levels; sequence rows for arm 4 and the warm start).
    -> hyp (H19-1 to H19-6), gate (Gate A1), kill, best, arms (per-arm means), perRun, spec, halts."""
    ix, A = index(rows), {a: runNames19(a, runs) for a in ARMS19}
    names, alpha = [n for v in A.values() for n in v], plan['alpha']
    coverage(ix, names, warm, seeds, sequences=A['a19open'])
    M = {m: runMeasures(ix, m, warm) for m in names + [warm]}
    col = lambda arm, f: [f(M[m]) for m in A[arm]]
    d4, sl = (lambda x: x['damage'][4]), (lambda x: x['waitSlope'])
    hyp = {h: dict(permTest(col(a, f), col(b, f)), n=[len(A[a]), len(A[b])], arms=[a, b])
           for h, a, b, f in (('H19_1', 'a19contest', 'a19credit', d4), ('H19_2', 'a19contest', 'a19fixed', sl))}
    prim = plan['families']['primary']
    for h, q in zip(prim, holm([hyp[h]['p'] for h in prim])):
        hyp[h].update(estimate=hyp[h]['diff'], pHolm=float(q), supported=bool(q < alpha and hyp[h]['diff'] > 0))
    w0 = float(np.mean([float(r['honestPooled']) for t in ('gL0T15', 'gL0T30') for r in ix.get(f'{warm}|plain{t}', [])]))
    b0 = {m: {(r['seed'], t): float(r['honestPooled']) for t in ('gL0T15', 'gL0T30') for r in ix.get(f'{m}|{t}', [])} for m in A['a19contest']}
    est = _m(col('a19contest', lambda x: x['cost0']))       # cost without liars: the warm start without guards minus arm 2
    hyp['H19_3'] = dict(estimate=est, margin=0.05, holds=bool(est <= 0.05), ci=hierBoot({'a': b0}, lambda d: w0 - float(d['a'].mean()), nboot),
                        raw=_m(col('a19contest', lambda x: x['raw0'])), constraint=True)
    spec = {m: specMeasures(ix, m, nNull) for m in A['a19open'] + [warm]} if A['a19open'] else {}
    if spec:
        d = [spec[m]['spec'][16] - spec[warm]['spec'][16] for m in A['a19open']]
        sf, slopes = signFlip(d), [spec[m]['specSlope'] for m in A['a19open']]
        hyp['H19_4'] = dict(estimate=sf['mean'], p=sf['p'], n=len(d), supported=bool(sf['p'] < alpha and sf['mean'] > 0),
                            slope=dict(estimate=_m(slopes), ci=_boot(slopes, nboot), warm=spec[warm]['specSlope']),
                            persistence=dict(estimate=_m([spec[m]['persistence'] for m in A['a19open']]), warm=spec[warm]['persistence']),
                            recovery={k: _m([spec[m]['recovery'][k] for m in A['a19open']]) for k in ('removed', 'returned', 'takeUp')},
                            killCriterion=True)
    drift = {}
    for a, ms in A.items():
        x = [M[m]['misreport'] - M[warm]['misreport'] for m in ms]
        drift[a] = dict(estimate=_m(x), ci=_boot(x, nboot), collapse=_m([M[m]['collapse'] for m in ms]))
    hyp['H19_5'] = dict(drift=drift, collapsed=[a for a, v in drift.items() if v['collapse'] > 0.5], secondary=True)
    gap = col('a19contest', lambda x: x['lifeGap'])
    sf = signFlip([-g for g in gap])
    hyp['H19_6'] = dict(estimate=_m(gap), ci=_boot(gap, nboot), p=sf['p'], secondary=True)
    gate = bool(all(hyp[h]['supported'] for h in prim) and hyp['H19_3']['holds'])
    arms = {a: {k: _m(col(a, (lambda x, k=k: x[k]))) for k in ('waitSlope', 'regret', 'contestsPerLie', 'contestsPerGuardLie',
            'timeToContest', 'guardedLies', 'misreport', 'cost0', 'raw0', 'gullibility', 'suppression', 'errorCorr', 'collaboration',
            'welfare', 'welfarePerKTok', 'lifeGap', 'collapse')} for a, ms in A.items() if ms}
    for a in arms:
        arms[a].update(damage={n: _m(col(a, lambda x, n=n: x['damage'][n])) for n in (2, 4, 6, 8)},
                       waitCurve={n: _m(col(a, lambda x, n=n: x['waitCurve'][n])) for n in (0, 2, 4, 6, 8)},
                       waitGrid={c: _m(col(a, lambda x, c=c: x['waitGrid'][c])) for c in M[warm]['waitGrid']},
                       held={k: _m(col(a, lambda x, k=k: x['held'][k])) for k in ('n8', 'n32', 'error', 'collude')})
    best = 'a19contest' if gate else max(arms, key=lambda a: arms[a]['damage'][4] if np.isfinite(arms[a]['damage'][4]) else -np.inf)
    return dict(hyp=hyp, gate=gate, kill=bool(not gate and not hyp.get('H19_4', {}).get('supported', False)), best=best,
                arms=arms, perRun=M, spec=spec, halts=haltCauses(halts))

## Report and figure

In [ ]:
#| export
def _g(x):
    return 'n/a' if x is None or not isinstance(x, (int, float)) or not np.isfinite(x) else f'{x:.3f}'


def writeRL19(path, S, meta, plan=RL19_PLAN):
    "report.md: H19-1 to H19-6, Gate A1, the kill criterion and the best arm, per-arm measures, arm 4, halts, and every plan item."
    h = S['hyp']
    L = ['# Phase 19: training the guard role with RL (contest credit)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| hypothesis | estimate | p (one-sided, exact) | Holm | verdict |', '|---|---|---|---|---|']
    for k in ('H19_1', 'H19_2'):
        L.append(f"| {k} (primary: {' vs '.join(h[k]['arms'])}) | {_g(h[k]['estimate'])} | {_g(h[k]['p'])} | {_g(h[k]['pHolm'])} | {h[k]['supported']} |")
    c = h['H19_3']
    L.append(f"| H19_3 (constraint: cost without liars against the warm start without guards, at most {c['margin']}) | {_g(c['estimate'])} "
             f"[{_g(c['ci'][0])}, {_g(c['ci'][1])}]; raw welfare {_g(c['raw'])} | | | holds: {c['holds']} |")
    if 'H19_4' in h:
        L.append(f"| H19_4 (kill criterion, specialization at 16) | {_g(h['H19_4']['estimate'])} | {_g(h['H19_4']['p'])} | | {h['H19_4']['supported']} |")
    L.append(f"| H19_6 (secondary, waiting at 15 minus 30 turns) | {_g(h['H19_6']['estimate'])} [{_g(h['H19_6']['ci'][0])}, {_g(h['H19_6']['ci'][1])}] | {_g(h['H19_6']['p'])} | | |")
    L += ['', f"**Gate A1:** {'passed' if S['gate'] else 'failed'}. **Kill criterion:** {'met (C3 and further RL of honest teams stop)' if S['kill'] else 'not met'}. "
          f"**Best arm:** {S['best']}.", '', '## Per arm (means over runs)', '',
          '| arm | damage won back at 2 / 4 / 6 / 8 liars | waiting slope | regret | contests per lie (guard-held) | time to contest | '
          'lies on a guarded clue | misreporting | cost without liars (raw) | gullibility | suppression | error correlation | '
          'collaboration | welfare (per 1k tokens) | collapse |', '|' + '---|' * 16]
    for a, v in S['arms'].items():
        L.append(f"| {ARM_LABEL[a]} | {' / '.join(_g(v['damage'][n]) for n in (2, 4, 6, 8))} | {_g(v['waitSlope'])} | {_g(v['regret'])} | "
                 f"{_g(v['contestsPerLie'])} ({_g(v['contestsPerGuardLie'])}) | {_g(v['timeToContest'])} | {_g(v['guardedLies'])} | "
                 f"{_g(v['misreport'])} | {_g(v['cost0'])} ({_g(v['raw0'])}) | {_g(v['gullibility'])} | {_g(v['suppression'])} | "
                 f"{_g(v['errorCorr'])} | {_g(v['collaboration'])} | "
                 f"{_g(v['welfare'])} ({_g(v['welfarePerKTok'])}) | {_g(v['collapse'])} |")
    L += ['', '## Waiting rate by liars of 16 (both lifetimes) and held-out damage won back', '',
          '| arm | 0 | 2 | 4 | 6 | 8 | 8 agents | 32 agents | mistaken members | colluding liars |', '|' + '---|' * 10]
    L += [f"| {ARM_LABEL[a]} | " + ' | '.join(_g(v['waitCurve'][n]) for n in (0, 2, 4, 6, 8)) + ' | ' +
          ' | '.join(_g(v['held'][k]) for k in ('n8', 'n32', 'error', 'collude')) + ' |' for a, v in S['arms'].items()]
    cells = sorted(next(iter(S['arms'].values()))['waitGrid']) if S['arms'] else []
    L += ['', '## Waiting rate by liars of 16 and subtask lifetime', '', '| arm | ' + ' | '.join(c[1:].replace('T', ' liars, ') + ' turns' for c in cells) + ' |',
          '|' + '---|' * (len(cells) + 1)] + [f"| {ARM_LABEL[a]} | " + ' | '.join(_g(v['waitGrid'][c]) for c in cells) + ' |' for a, v in S['arms'].items()]
    L += ['', '## H19-5: misreporting minus the warm start\'s, and collapse', '', '| arm | change | 95% interval | boards below a third of the answers |', '|---|---|---|---|']
    L += [f"| {ARM_LABEL[a]} | {_g(v['estimate'])} | [{_g(v['ci'][0])}, {_g(v['ci'][1])}] | {_g(v['collapse'])} |" for a, v in h['H19_5']['drift'].items()]
    if S['spec']:
        L += ['', "## Arm 4 over sequences of 6 boards: specialization (Part B's variance ratio minus its null), division of labour "
              "(Gorelick's index), role persistence and recovery", '', '| model | specialization at 8 | 16 | 32 | slope over log2 size | '
              'division of labour at 8 | 16 | 32 | persistence | removed | returned | take-up |', '|' + '---|' * 12]
        L += [f"| {m} | " + ' | '.join(_g(v['spec'][N]) for N in (8, 16, 32)) + f" | {_g(v['specSlope'])} | " +
              ' | '.join(_g(v['dol'][N]) for N in (8, 16, 32)) + f" | {_g(v['persistence'])} | " +
              ' | '.join(_g(v['recovery'][k]) for k in ('removed', 'returned', 'takeUp')) + ' |' for m, v in S['spec'].items()]
    L += ['', '## Halts by cause', ''] + [f"- {k}: {', '.join(v)}" for k, v in S['halts'].items()] + ['', '## RL19_PLAN', '']
    L += [f"- {k} ({v['proposal']}): {v['value']!r}, confirmed {v['confirmed']}" for k, v in plan['decisions'].items()]
    L += [f"- {k}: {json.dumps(v, default=str)}" for k, v in plan.items() if k != 'decisions']
    open(path, 'w').write('\n'.join(L) + '\n')


def figRL19(out, S):
    "rl19.png: damage won back at 4 liars per arm (runs and mean), waiting rate against liars per arm, arm 4's specialization by team size."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    col = dict(a19credit='#7f7f7f', a19contest='#1f6391', a19fixed='#c0392b', a19open='#2e8b57')
    for k, a in enumerate(S['arms']):
        v = [S['perRun'][m]['damage'][4] for m in runNames19(a) if m in S['perRun']]
        ax[0].plot([k] * len(v), v, 'o', color=col[a], alpha=0.5)
        ax[0].plot([k - 0.25, k + 0.25], [S['arms'][a]['damage'][4]] * 2, '-', color=col[a], lw=2)
        ax[1].plot(list(S['arms'][a]['waitCurve']), list(S['arms'][a]['waitCurve'].values()), '-o', color=col[a], label=ARM_LABEL[a])
    ax[0].set(xticks=range(len(S['arms'])), xticklabels=[ARM_LABEL[a].split(' ')[0] for a in S['arms']], xlabel='arm', ylabel='damage won back (4 of 16 liars)')
    ax[1].set(xlabel='liars of 16', ylabel='waiting rate')
    ax[1].legend(fontsize=7)
    for m, v in S['spec'].items():
        ax[2].plot([8, 16, 32], [v['spec'][N] for N in (8, 16, 32)], 'k-o' if m == 'warm' else '-', color=None if m == 'warm' else col['a19open'], lw=2 if m == 'warm' else 0.8)
    ax[2].set(xscale='log', xticks=[8, 16, 32], xticklabels=['8', '16', '32'], xlabel='team size', ylabel='specialization (arm 4; black: warm)')
    fig.tight_layout()
    fig.savefig(f'{out}/rl19.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

## Tests

In [ ]:
# test_rl19_report (AC-7): on synthetic rows with known effects the exact tests and Holm recover them (the smallest p of 8 + 8
# runs, 1/12870; arm 4's 5 runs, 1/32), H19-3 holds, Gate A1 passes; without effects nothing is supported and the kill
# criterion is met; incomplete rows are refused; the report names every hypothesis and plan item
import os, tempfile
from explRL.rl19_report import *
from explRL.rl19_synth import syntheticRows19
S = rl19Summaries(syntheticRows19(), nboot=200, nNull=30, halts={'a19credit_s1': None, 'a19open_s2': "answers per board below a third (steps 5-7)"})
h = S['hyp']
assert h['H19_1']['p'] == 1 / 12870 and h['H19_2']['p'] == 1 / 12870 and h['H19_1']['pHolm'] == h['H19_2']['pHolm'] == 2 / 12870
assert h['H19_1']['supported'] and h['H19_2']['supported'] and abs(h['H19_1']['estimate'] - 0.25) < 0.03
assert abs(h['H19_2']['estimate'] - 0.03) < 0.003 and h['H19_3']['holds'] and abs(h['H19_3']['estimate'] - 0.01) < 0.003
assert h['H19_3']['ci'][0] < h['H19_3']['estimate'] < h['H19_3']['ci'][1]
assert h['H19_4']['p'] == 1 / 32 and h['H19_4']['supported'] and S['gate'] and not S['kill'] and S['best'] == 'a19contest'
assert h['H19_6']['estimate'] < 0 and h['H19_6']['p'] == 1 / 256 and h['H19_5']['collapsed'] == []
assert abs(S['arms']['a19contest']['regret'] - (0.32 - 0.355)) < 0.005      # the best fixed threshold per liar count (K 3 at each) against arm 2's welfare
assert h['H19_6']['secondary'] and 'H19_6' in RL19_PLAN['families']['secondary'] and S['arms']['a19contest']['waitGrid']['L8T15'] > S['arms']['a19contest']['waitGrid']['L0T15']
S3 = rl19Summaries(syntheticRows19(cost=0.08), nboot=100, nNull=30)
assert not S3['hyp']['H19_3']['holds'] and abs(S3['hyp']['H19_3']['estimate'] - 0.08) < 0.003 and not S3['gate']   # a cost above 0.05 fails the gate
assert S['halts'] == {'none': ['a19credit_s1'], 'answers': ['a19open_s2']}
S0 = rl19Summaries(syntheticRows19(damage=0.0, slope=(0.01, 0.01), spec=False, seed=1), nboot=100, nNull=30)
assert not any(S0['hyp'][k]['supported'] for k in ('H19_1', 'H19_2', 'H19_4')) and not S0['gate'] and S0['kill']
assert S0['best'] in ARMS19
bad = [r for r in syntheticRows19(seeds=[900, 901]) if r['level'] != 'a19fixed_s3|gL6T15']
try:
    rl19Summaries(bad, seeds=2)
    raise AssertionError('rows without the full grid accepted')
except ValueError as e:
    assert 'a19fixed_s3|gL6T15' in str(e)
with tempfile.TemporaryDirectory() as d:
    writeRL19(f'{d}/report.md', S, dict(mode='test'))
    figRL19(d, S)
    txt = open(f'{d}/report.md').read()
    assert os.path.getsize(f'{d}/rl19.png') > 0 and all(k in txt for k in [f'H19_{i}' for i in range(1, 7)] + [f'P19_{i}' for i in range(1, 8)])
    assert all(f'- {k}:' in txt for k in RL19_PLAN if k != 'decisions') and 'Gate A1:** passed' in txt

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()